# Lab — Enterprise visual quality inspection

Build an auditable five-class quality gate from pixels to learned representations. You will create and profile a multi-source dataset, train a CNN from scratch, reuse two real pretrained encoders, inspect their embedding geometry, inject a deployment shift, analyse failures, and save an enterprise decision artifact.

**Classes:** Normal · Surface Defect · Structural Defect · Contamination · Unknown / Ambiguous

**Decision boundary:** this is a learning system on synthetic data, not a validated inspection model. The final policy must preserve a human-review route.

![Enterprise vision pipeline](assets/enterprise-vision-pipeline.svg)

## 0. Objectives, success criteria, and experiment contract

By the end of this notebook you should be able to explain:

- what each image tensor dimension and normalization step means;
- how convolution and receptive field create a visual hierarchy;
- why source-aware splitting and duplicate checks precede training;
- what frozen transfer and partial fine-tuning change;
- what nearest neighbours and PCA reveal—and what they do not;
- how clean performance differs from shifted performance; and
- why a model score becomes an enterprise decision only after thresholds, abstention, and monitoring.

The default configuration is deliberately CPU-friendly. Set `CV_FULL_RUN=1` before launching Jupyter for a larger generated dataset and longer training. Both modes execute the same real pipeline; neither mocks model outputs.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import sklearn
import torch
import torchvision
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.preprocessing import normalize
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    ResNet18_Weights,
    convnext_tiny,
    resnet18,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

cwd = Path.cwd().resolve()
relative_course = Path("curriculum/beginner/01-modern-computer-vision-foundations")
COURSE_DIR = cwd / relative_course if (cwd / relative_course).exists() else cwd
ARTIFACT_DIR = COURSE_DIR / ".artifacts" / "enterprise_quality_inspection"
DATA_DIR = ARTIFACT_DIR / "dataset"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FULL_RUN = os.getenv("CV_FULL_RUN", "0") == "1"
DEVICE = torch.device("cpu")  # stable cross-platform teaching baseline

@dataclass(frozen=True)
class Config:
    image_size: int = 96
    pretrained_crop: int = 128
    train_per_class: int = 48 if FULL_RUN else 20
    val_per_class: int = 16 if FULL_RUN else 8
    test_per_class: int = 20 if FULL_RUN else 10
    scratch_epochs: int = 12 if FULL_RUN else 5
    partial_epochs: int = 4 if FULL_RUN else 1
    batch_size: int = 16
    review_threshold: float = 0.70

CFG = Config()
CLASS_NAMES = [
    "Normal",
    "Surface Defect",
    "Structural Defect",
    "Contamination",
    "Unknown / Ambiguous",
]
CLASS_SLUGS = ["normal", "surface_defect", "structural_defect", "contamination", "unknown_ambiguous"]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

versions = pd.Series({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pillow": PIL.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(DEVICE),
    "full_run": FULL_RUN,
})
display(versions.to_frame("value"))
print("Artifacts:", ARTIFACT_DIR)

## 1. Generate a controlled multi-source dataset

Public industrial benchmarks are valuable, but they may have noncommercial licences or a different label protocol. Here we generate a deterministic dataset that makes the experiment auditable:

- `Factory_A` and `Factory_B` supply training/validation data.
- `Factory_C` is held out until final evaluation.
- known-source backgrounds contain a label-correlated tint, creating a learnable shortcut;
- the held-out source removes that correlation;
- one exact image is deliberately copied from train to validation so the profile must catch it.

This design is a teaching instrument. Synthetic performance is not evidence of real-world readiness.

In [ ]:
def make_component(label_id: int, source: str, sample_seed: int, size: int) -> Image.Image:
    rng = np.random.default_rng(sample_seed)
    yy, xx = np.mgrid[0:size, 0:size]

    source_bias = {"Factory_A": -10, "Factory_B": 8, "Factory_C": 0}[source]
    shortcut = (label_id - 2) * 12 if source != "Factory_C" else 0
    base = np.clip(142 + source_bias + shortcut + 0.10 * (xx - size / 2), 55, 220)
    noise = rng.normal(0, 5, (size, size))
    plate = np.clip(base + noise, 0, 255).astype(np.uint8)
    rgb = np.stack([plate, np.clip(plate + 3, 0, 255), np.clip(plate + 7, 0, 255)], axis=-1).astype(np.uint8)
    image = Image.fromarray(rgb, mode="RGB")
    draw = ImageDraw.Draw(image, "RGBA")

    margin = size // 8
    draw.rounded_rectangle((margin, margin, size - margin, size - margin), radius=10, outline=(35, 45, 55, 220), width=3)
    for x, y in [(margin + 8, margin + 8), (size - margin - 8, margin + 8), (margin + 8, size - margin - 8), (size - margin - 8, size - margin - 8)]:
        draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=(55, 65, 75, 230))

    if label_id == 1:  # surface scratches
        for _ in range(3):
            x0 = int(rng.integers(margin + 8, size // 2))
            y0 = int(rng.integers(margin + 8, size - margin - 8))
            x1 = int(rng.integers(size // 2, size - margin - 5))
            draw.line((x0, y0, x1, y0 + int(rng.integers(-7, 8))), fill=(235, 235, 225, 235), width=2)
            draw.line((x0, y0 + 2, x1, y0 + int(rng.integers(-7, 8)) + 2), fill=(50, 55, 60, 170), width=1)
    elif label_id == 2:  # structural crack
        points = [(margin + 10, size // 3)]
        for step in range(1, 6):
            points.append((margin + 10 + step * (size - 2 * margin - 20) // 5, size // 3 + int(rng.integers(-11, 12))))
        draw.line(points, fill=(25, 20, 25, 245), width=4)
        draw.line([(x, y + 2) for x, y in points], fill=(225, 205, 190, 130), width=1)
    elif label_id == 3:  # contamination
        for _ in range(7):
            x = int(rng.integers(margin + 8, size - margin - 8))
            y = int(rng.integers(margin + 8, size - margin - 8))
            radius = int(rng.integers(3, 9))
            colour = (95, int(rng.integers(60, 105)), 30, int(rng.integers(100, 190)))
            draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=colour)
    elif label_id == 4:  # intentionally ambiguous signal
        x0, y0 = margin + 12, int(rng.integers(size // 3, 2 * size // 3))
        draw.line((x0, y0, size - margin - 12, y0 + int(rng.integers(-5, 6))), fill=(70, 65, 70, 110), width=2)
        draw.ellipse((size // 2 - 7, size // 2 - 5, size // 2 + 8, size // 2 + 6), fill=(100, 75, 40, 70))
        draw.rectangle((size // 2 - 11, margin + 2, size // 2 + 11, size - margin - 2), fill=(210, 210, 210, 35))

    if source == "Factory_B":
        image = ImageEnhance.Contrast(image).enhance(1.08)
    elif source == "Factory_C":
        image = ImageEnhance.Color(image).enhance(0.82)
    return image


def generate_dataset() -> pd.DataFrame:
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    rows = []
    split_spec = {
        "train": (["Factory_A", "Factory_B"], CFG.train_per_class),
        "val": (["Factory_A", "Factory_B"], CFG.val_per_class),
        "test": (["Factory_C"], CFG.test_per_class),
    }
    running_id = 0
    for split, (sources, per_class) in split_spec.items():
        for label_id, (label, slug) in enumerate(zip(CLASS_NAMES, CLASS_SLUGS)):
            for index in range(per_class):
                source = sources[index % len(sources)]
                sample_id = f"{split}_{running_id:04d}"
                path = DATA_DIR / split / slug / f"{sample_id}.png"
                path.parent.mkdir(parents=True, exist_ok=True)
                seed = SEED * 10_000 + running_id
                make_component(label_id, source, seed, CFG.image_size).save(path)
                rows.append({
                    "sample_id": sample_id,
                    "path": str(path),
                    "split": split,
                    "source": source,
                    "label": label,
                    "label_id": label_id,
                    "duplicate_of": None,
                })
                running_id += 1

    frame = pd.DataFrame(rows)
    original = frame.query("split == 'train' and label == 'Normal'").iloc[0]
    duplicate_path = DATA_DIR / "val" / "normal" / "val_deliberate_duplicate.png"
    duplicate_path.write_bytes(Path(original.path).read_bytes())
    duplicate = {
        "sample_id": "val_deliberate_duplicate",
        "path": str(duplicate_path),
        "split": "val",
        "source": "Factory_B",
        "label": "Normal",
        "label_id": 0,
        "duplicate_of": original.sample_id,
    }
    return pd.concat([frame, pd.DataFrame([duplicate])], ignore_index=True)


raw_df = generate_dataset()
print(f"Generated {len(raw_df):,} records under {DATA_DIR}")
display(pd.crosstab([raw_df["split"], raw_df["source"]], raw_df["label"]))

## 2. Profile before modelling

A file count is not a data profile. We verify readability, dimensions, channels, numerical extrema, and hashes. The exact duplicate check uses file bytes here because the deliberate copy is byte-identical. Real pipelines should also compare decoded pixels and use perceptual or embedding similarity for resized/re-encoded near duplicates.

In [ ]:
def inspect_file(path_value: str) -> pd.Series:
    path = Path(path_value)
    payload = path.read_bytes()
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"))
    return pd.Series({
        "sha256": hashlib.sha256(payload).hexdigest(),
        "width": array.shape[1],
        "height": array.shape[0],
        "channels": array.shape[2],
        "pixel_min": int(array.min()),
        "pixel_max": int(array.max()),
    })


profile = raw_df["path"].apply(inspect_file)
profiled_df = pd.concat([raw_df, profile], axis=1)
duplicate_rows = profiled_df[profiled_df.duplicated("sha256", keep=False)].sort_values("sha256")
display(profiled_df.groupby("split")[["width", "height", "channels", "pixel_min", "pixel_max"]].agg(["min", "max"]))
display(duplicate_rows[["sample_id", "split", "source", "label", "duplicate_of", "sha256"]])

cross_split_duplicate_hashes = set(
    duplicate_rows.groupby("sha256")["split"].nunique().loc[lambda values: values > 1].index
)
remove_mask = (profiled_df["split"] != "train") & profiled_df["sha256"].isin(cross_split_duplicate_hashes)
clean_df = profiled_df.loc[~remove_mask].reset_index(drop=True)
assert not clean_df.groupby("sha256")["split"].nunique().gt(1).any()
assert set(clean_df.query("split == 'test'")["source"]) == {"Factory_C"}
print(f"Removed {int(remove_mask.sum())} cross-split duplicate; {len(clean_df)} clean records remain.")

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(10, 12))
for row, class_name in enumerate(CLASS_NAMES):
    examples = clean_df.query("label == @class_name").sample(4, random_state=SEED + row)
    for axis, (_, record) in zip(axes[row], examples.iterrows()):
        axis.imshow(Image.open(record.path))
        axis.set_title(f"{class_name}\n{record.source} · {record.split}", fontsize=8)
        axis.axis("off")
fig.suptitle("Profiled samples: inspect labels and sources before training", y=1.01)
plt.tight_layout()

## 3. Image contracts, convolution, and receptive field

Pillow/NumPy use `H × W × C`; PyTorch convolution uses `N × C × H × W`. The next cell makes the conversion explicit and applies three fixed kernels. These responses are not learned. They establish the mechanics that the scratch CNN will learn from data.

![CNN representation hierarchy](assets/cnn-representation-hierarchy.svg)

In [ ]:
sample_path = clean_df.query("label == 'Structural Defect'").iloc[0].path
sample_hwc = np.asarray(Image.open(sample_path).convert("RGB"), dtype=np.float32) / 255.0
sample_nchw = torch.from_numpy(sample_hwc).permute(2, 0, 1).unsqueeze(0)
gray = sample_nchw.mean(dim=1, keepdim=True)

kernels = {
    "identity": [[0, 0, 0], [0, 1, 0], [0, 0, 0]],
    "vertical edge": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "sharpen": [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}
responses = {}
for name, values in kernels.items():
    kernel = torch.tensor(values, dtype=torch.float32).view(1, 1, 3, 3)
    responses[name] = F.conv2d(gray, kernel, padding=1).squeeze().numpy()

print("NumPy contract:", sample_hwc.shape, sample_hwc.dtype, (sample_hwc.min(), sample_hwc.max()))
print("PyTorch contract:", tuple(sample_nchw.shape), sample_nchw.dtype)
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(sample_hwc)
axes[0].set_title("RGB input")
for axis, (name, response) in zip(axes[1:], responses.items()):
    axis.imshow(response, cmap="coolwarm")
    axis.set_title(name)
for axis in axes:
    axis.axis("off")
plt.tight_layout()

In [ ]:
layers = [
    {"name": "conv1", "kernel": 3, "stride": 1, "dilation": 1},
    {"name": "pool1", "kernel": 2, "stride": 2, "dilation": 1},
    {"name": "conv2", "kernel": 3, "stride": 1, "dilation": 1},
    {"name": "pool2", "kernel": 2, "stride": 2, "dilation": 1},
    {"name": "conv3", "kernel": 3, "stride": 1, "dilation": 1},
]
receptive_field, jump = 1, 1
rf_rows = []
for layer in layers:
    receptive_field += (layer["kernel"] - 1) * layer["dilation"] * jump
    jump *= layer["stride"]
    rf_rows.append({**layer, "effective_jump": jump, "receptive_field": receptive_field})
display(pd.DataFrame(rf_rows))

## 4. Leakage-safe datasets and preprocessing

The scratch model uses moderate, label-preserving augmentation on training data only. Evaluation is deterministic. Pretrained encoders later use the preprocessing bundled with their official weight objects, preventing an easy but damaging contract mismatch.

In [ ]:
scratch_train_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.25] * 3),
])
scratch_eval_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.25] * 3),
])

class InspectionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform, shift_fn=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.shift_fn = shift_fn

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row.path).convert("RGB")
        if self.shift_fn is not None:
            image = self.shift_fn(image)
        return self.transform(image), int(row.label_id), index


splits = {name: clean_df.query("split == @name").reset_index(drop=True) for name in ["train", "val", "test"]}
train_ds = InspectionDataset(splits["train"], scratch_train_transform)
val_ds = InspectionDataset(splits["val"], scratch_eval_transform)
test_ds = InspectionDataset(splits["test"], scratch_eval_transform)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)

images, labels, indices = next(iter(train_loader))
print("Batch contract:", tuple(images.shape), images.dtype, "labels:", tuple(labels.shape))

## 5. Baseline — train a compact CNN from scratch

A scratch model is essential evidence. It tells us whether the task is learnable with local patterns and whether pretrained features add value relative to their cost. The architecture is intentionally small: three convolution blocks, global average pooling, and a linear head.

In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self, class_count: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, 3, padding=1), nn.BatchNorm2d(24), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(96, class_count)

    def forward(self, inputs):
        return self.classifier(self.features(inputs).flatten(1))


def train_model(model, train_data, val_data, epochs, learning_rate, weight_decay=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    history, best_state, best_f1 = [], copy.deepcopy(model.state_dict()), -1.0
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch, targets, _ in train_data:
            batch, targets = batch.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch), targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(targets)
        val_result = predict_torch(model, val_data)
        val_f1 = f1_score(val_result["y_true"], val_result["y_pred"], average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": train_loss / len(train_data.dataset), "val_macro_f1": val_f1})
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), time.perf_counter() - started


@torch.inference_mode()
def predict_torch(model, loader):
    model.eval()
    labels, probabilities, row_indices = [], [], []
    started = time.perf_counter()
    for batch, targets, indices in loader:
        logits = model(batch.to(DEVICE))
        probabilities.append(logits.softmax(dim=1).cpu().numpy())
        labels.append(targets.numpy())
        row_indices.append(indices.numpy())
    probs = np.concatenate(probabilities)
    return {
        "y_true": np.concatenate(labels),
        "y_pred": probs.argmax(axis=1),
        "probs": probs,
        "row_indices": np.concatenate(row_indices),
        "seconds": time.perf_counter() - started,
    }


def quality_metrics(y_true, probs, threshold=CFG.review_threshold):
    y_pred = probs.argmax(axis=1)
    is_defect = np.asarray(y_true) != 0
    predicted_defect = y_pred != 0
    review = (probs.max(axis=1) < threshold) | (y_pred == CLASS_TO_ID["Unknown / Ambiguous"])
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "defect_recall": recall_score(is_defect, predicted_defect, zero_division=0),
        "normal_recall": recall_score(~is_defect, ~predicted_defect, zero_division=0),
        "review_rate": review.mean(),
    }


scratch_model = ScratchCNN(len(CLASS_NAMES))
scratch_parameters = sum(parameter.numel() for parameter in scratch_model.parameters())
scratch_model, scratch_history, scratch_train_seconds = train_model(
    scratch_model, train_loader, val_loader, CFG.scratch_epochs, learning_rate=2e-3
)
display(scratch_history)
print(f"Scratch parameters: {scratch_parameters:,}; train time: {scratch_train_seconds:.2f}s")

In [ ]:
scratch_clean = predict_torch(scratch_model, test_loader)
display(pd.Series(quality_metrics(scratch_clean["y_true"], scratch_clean["probs"]), name="scratch_clean").to_frame())
print(classification_report(scratch_clean["y_true"], scratch_clean["y_pred"], target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    scratch_clean["y_true"], scratch_clean["y_pred"], display_labels=CLASS_NAMES, xticks_rotation=35, cmap="Blues"
)
plt.title("Scratch CNN · held-out Factory_C")
plt.tight_layout()

## 6. Real pretrained encoders: ResNet-18 and ConvNeXt-Tiny

The next experiment downloads official torchvision weights, freezes every encoder parameter, extracts one vector per image, and fits a multinomial logistic-regression probe. The probe is deliberately simple: representation quality must do most of the work.

![Transfer learning options](assets/transfer-learning.svg)

In [ ]:
def build_frozen_encoder(name: str):
    if name == "resnet18":
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights)
        dimension = model.fc.in_features
        model.fc = nn.Identity()
    elif name == "convnext_tiny":
        weights = ConvNeXt_Tiny_Weights.DEFAULT
        model = convnext_tiny(weights=weights)
        dimension = model.classifier[-1].in_features
        model.classifier[-1] = nn.Identity()
    else:
        raise ValueError(name)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval().to(DEVICE)
    return model, weights, dimension


@torch.inference_mode()
def extract_features(model, weights, frame, shift_fn=None):
    transform = weights.transforms(crop_size=CFG.pretrained_crop, resize_size=CFG.pretrained_crop + 16)
    loader = DataLoader(
        InspectionDataset(frame, transform, shift_fn=shift_fn),
        batch_size=max(4, CFG.batch_size // 2),
        shuffle=False,
    )
    features, labels = [], []
    started = time.perf_counter()
    for batch, targets, _ in loader:
        features.append(model(batch.to(DEVICE)).cpu().numpy())
        labels.append(targets.numpy())
    return np.concatenate(features), np.concatenate(labels), time.perf_counter() - started


all_ordered = pd.concat([splits["train"], splits["val"], splits["test"]], ignore_index=True)
boundaries = {
    "train": slice(0, len(splits["train"])),
    "val": slice(len(splits["train"]), len(splits["train"]) + len(splits["val"])),
    "test": slice(len(splits["train"]) + len(splits["val"]), len(all_ordered)),
}
encoders, feature_store, probes, extraction_seconds = {}, {}, {}, {}
for encoder_name in ["resnet18", "convnext_tiny"]:
    model, weights, dimension = build_frozen_encoder(encoder_name)
    features, labels, elapsed = extract_features(model, weights, all_ordered)
    probe = LogisticRegression(max_iter=600, class_weight="balanced", random_state=SEED)
    probe.fit(features[boundaries["train"]], labels[boundaries["train"]])
    encoders[encoder_name] = (model, weights, dimension)
    feature_store[encoder_name] = features
    probes[encoder_name] = probe
    extraction_seconds[encoder_name] = elapsed
    print(f"{encoder_name}: {dimension}-D embeddings for {len(features)} images in {elapsed:.2f}s")

In [ ]:
frozen_clean = {}
for encoder_name, probe in probes.items():
    test_slice = boundaries["test"]
    probabilities = probe.predict_proba(feature_store[encoder_name][test_slice])
    frozen_clean[encoder_name] = {
        "y_true": splits["test"]["label_id"].to_numpy(),
        "y_pred": probabilities.argmax(axis=1),
        "probs": probabilities,
    }
    display(pd.Series(quality_metrics(frozen_clean[encoder_name]["y_true"], probabilities), name=encoder_name).to_frame())

## 7. Inspect representation geometry

A useful classifier score can still hide a brittle embedding space. We inspect `ResNet-18` nearest neighbours and a two-dimensional PCA projection. Neighbours should share defect evidence rather than only background. PCA is fitted on training embeddings and then applied to validation/test embeddings, avoiding leakage.

In [ ]:
encoder_name = "resnet18"
train_features = normalize(feature_store[encoder_name][boundaries["train"]])
test_features = normalize(feature_store[encoder_name][boundaries["test"]])
query_indices = [0, len(splits["test"]) // 2, len(splits["test"]) - 1]
fig, axes = plt.subplots(len(query_indices), 4, figsize=(11, 8))
for row, query_index in enumerate(query_indices):
    similarities = train_features @ test_features[query_index]
    neighbour_indices = similarities.argsort()[-3:][::-1]
    query = splits["test"].iloc[query_index]
    axes[row, 0].imshow(Image.open(query.path))
    axes[row, 0].set_title(f"Query\n{query.label}", fontsize=8)
    for column, neighbour_index in enumerate(neighbour_indices, start=1):
        neighbour = splits["train"].iloc[neighbour_index]
        axes[row, column].imshow(Image.open(neighbour.path))
        axes[row, column].set_title(f"{neighbour.label}\nsim={similarities[neighbour_index]:.2f}", fontsize=8)
    for axis in axes[row]:
        axis.axis("off")
fig.suptitle("ResNet-18 embedding nearest neighbours", y=1.01)
plt.tight_layout()

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
train_2d = pca.fit_transform(feature_store[encoder_name][boundaries["train"]])
test_2d = pca.transform(feature_store[encoder_name][boundaries["test"]])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for axis, points, frame, title in [
    (axes[0], train_2d, splits["train"], "Known-source train"),
    (axes[1], test_2d, splits["test"], "Held-out Factory_C"),
]:
    for label_id, class_name in enumerate(CLASS_NAMES):
        mask = frame["label_id"].to_numpy() == label_id
        axis.scatter(points[mask, 0], points[mask, 1], s=24, alpha=0.75, label=class_name)
    axis.set_title(title)
    axis.set_xlabel("PC1")
    axis.grid(alpha=0.2)
axes[0].set_ylabel("PC2")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle(f"PCA fitted on training embeddings · explained variance {pca.explained_variance_ratio_.sum():.1%}")
plt.tight_layout()

## 8. Partial fine-tuning

Frozen transfer asks whether the existing representation is already sufficient. Partial fine-tuning adapts the final residual stage and classification head using a small learning rate. This increases flexibility and risk: a small dataset can overwrite useful general features or reinforce its shortcuts.

In [ ]:
resnet_weights = ResNet18_Weights.DEFAULT
resnet_transform = resnet_weights.transforms(crop_size=CFG.pretrained_crop, resize_size=CFG.pretrained_crop + 16)
transfer_train_loader = DataLoader(
    InspectionDataset(splits["train"], resnet_transform),
    batch_size=max(4, CFG.batch_size // 2),
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
transfer_val_loader = DataLoader(InspectionDataset(splits["val"], resnet_transform), batch_size=8, shuffle=False)
transfer_test_loader = DataLoader(InspectionDataset(splits["test"], resnet_transform), batch_size=8, shuffle=False)

partial_model = resnet18(weights=resnet_weights)
partial_model.fc = nn.Linear(partial_model.fc.in_features, len(CLASS_NAMES))
for parameter in partial_model.parameters():
    parameter.requires_grad = False
for parameter in partial_model.layer4.parameters():
    parameter.requires_grad = True
for parameter in partial_model.fc.parameters():
    parameter.requires_grad = True
partial_trainable = sum(parameter.numel() for parameter in partial_model.parameters() if parameter.requires_grad)
partial_model, partial_history, partial_train_seconds = train_model(
    partial_model, transfer_train_loader, transfer_val_loader, CFG.partial_epochs, learning_rate=3e-4
)
partial_clean = predict_torch(partial_model, transfer_test_loader)
display(partial_history)
display(pd.Series(quality_metrics(partial_clean["y_true"], partial_clean["probs"]), name="partial_resnet18").to_frame())
print(f"Trainable parameters: {partial_trainable:,}; train time: {partial_train_seconds:.2f}s")

## 9. Stress test a deployment shift

`Factory_C` was already a source shift. We now add a capture degradation: lower brightness/contrast, blur, and a downsample-upsample cycle. This is not a universal robustness benchmark; it is a reproducible hypothesis about one plausible operational failure.

In [ ]:
def deployment_shift(image: Image.Image) -> Image.Image:
    shifted = ImageEnhance.Brightness(image).enhance(0.72)
    shifted = ImageEnhance.Contrast(shifted).enhance(0.78)
    shifted = shifted.filter(ImageFilter.GaussianBlur(radius=1.1))
    width, height = shifted.size
    shifted = shifted.resize((width // 2, height // 2), Image.Resampling.BILINEAR)
    return shifted.resize((width, height), Image.Resampling.BILINEAR)


shift_examples = splits["test"].groupby("label", sort=False).head(1)
fig, axes = plt.subplots(len(shift_examples), 2, figsize=(6, 12))
for row, (_, record) in enumerate(shift_examples.iterrows()):
    clean_image = Image.open(record.path).convert("RGB")
    axes[row, 0].imshow(clean_image)
    axes[row, 1].imshow(deployment_shift(clean_image))
    axes[row, 0].set_ylabel(record.label)
    axes[row, 0].set_title("clean")
    axes[row, 1].set_title("shifted")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()

In [ ]:
shifted_scratch_loader = DataLoader(
    InspectionDataset(splits["test"], scratch_eval_transform, shift_fn=deployment_shift),
    batch_size=CFG.batch_size,
    shuffle=False,
)
scratch_shift = predict_torch(scratch_model, shifted_scratch_loader)

frozen_shift = {}
for encoder_name, (model, weights, _) in encoders.items():
    shifted_features, shifted_labels, elapsed = extract_features(model, weights, splits["test"], shift_fn=deployment_shift)
    probabilities = probes[encoder_name].predict_proba(shifted_features)
    frozen_shift[encoder_name] = {
        "y_true": shifted_labels,
        "y_pred": probabilities.argmax(axis=1),
        "probs": probabilities,
        "seconds": elapsed,
    }

shifted_partial_loader = DataLoader(
    InspectionDataset(splits["test"], resnet_transform, shift_fn=deployment_shift), batch_size=8, shuffle=False
)
partial_shift = predict_torch(partial_model, shifted_partial_loader)

## 10. Compare models on the evidence that matters

The table includes at least three model strategies and keeps clean and shifted metrics separate. Parameter counts are not all trainable parameters: frozen encoders still carry storage and inference cost. Timings are local measurements for relative context, not deployment benchmarks.

In [ ]:
clean_results = {
    "Scratch CNN": scratch_clean,
    "Frozen ResNet-18 + probe": frozen_clean["resnet18"],
    "Frozen ConvNeXt-Tiny + probe": frozen_clean["convnext_tiny"],
    "Partial ResNet-18": partial_clean,
}
shift_results = {
    "Scratch CNN": scratch_shift,
    "Frozen ResNet-18 + probe": frozen_shift["resnet18"],
    "Frozen ConvNeXt-Tiny + probe": frozen_shift["convnext_tiny"],
    "Partial ResNet-18": partial_shift,
}
model_costs = {
    "Scratch CNN": {"parameters": scratch_parameters, "fit_seconds": scratch_train_seconds},
    "Frozen ResNet-18 + probe": {
        "parameters": sum(p.numel() for p in encoders["resnet18"][0].parameters()),
        "fit_seconds": extraction_seconds["resnet18"],
    },
    "Frozen ConvNeXt-Tiny + probe": {
        "parameters": sum(p.numel() for p in encoders["convnext_tiny"][0].parameters()),
        "fit_seconds": extraction_seconds["convnext_tiny"],
    },
    "Partial ResNet-18": {"parameters": sum(p.numel() for p in partial_model.parameters()), "fit_seconds": partial_train_seconds},
}

comparison_rows = []
for name in clean_results:
    clean_metrics = quality_metrics(clean_results[name]["y_true"], clean_results[name]["probs"])
    shift_metrics = quality_metrics(shift_results[name]["y_true"], shift_results[name]["probs"])
    comparison_rows.append({
        "model": name,
        "clean_accuracy": clean_metrics["accuracy"],
        "clean_macro_f1": clean_metrics["macro_f1"],
        "clean_defect_recall": clean_metrics["defect_recall"],
        "shift_macro_f1": shift_metrics["macro_f1"],
        "shift_defect_recall": shift_metrics["defect_recall"],
        "f1_drop": clean_metrics["macro_f1"] - shift_metrics["macro_f1"],
        "review_rate": clean_metrics["review_rate"],
        "parameters_m": model_costs[name]["parameters"] / 1e6,
        "fit_or_extract_seconds": model_costs[name]["fit_seconds"],
    })
comparison = pd.DataFrame(comparison_rows).sort_values(["shift_defect_recall", "shift_macro_f1"], ascending=False)
display(comparison.style.format({column: "{:.3f}" for column in comparison.columns if column != "model"}))

In [ ]:
source_rows = []
for name, result in clean_results.items():
    for source, positions in splits["test"].groupby("source").groups.items():
        positions = np.asarray(list(positions))
        metrics = quality_metrics(result["y_true"][positions], result["probs"][positions])
        source_rows.append({"model": name, "source": source, **metrics})
display(pd.DataFrame(source_rows))

## 11. Failure gallery and abstention policy

We use frozen ResNet-18 for the policy exercise because it has a simple separation between reusable encoder and auditable linear probe. The threshold is selected on validation data only. `Unknown / Ambiguous` is always routed to review. A real policy would use calibrated probabilities, explicit error costs, reviewer capacity, and repeated temporal validation.

![Failure analysis loop](assets/cv-failure-analysis.svg)

In [ ]:
reference_name = "Frozen ResNet-18 + probe"
val_probs = probes["resnet18"].predict_proba(feature_store["resnet18"][boundaries["val"]])
val_true = splits["val"]["label_id"].to_numpy()
threshold_rows = []
for threshold in np.linspace(0.40, 0.95, 12):
    prediction = val_probs.argmax(axis=1)
    review = (val_probs.max(axis=1) < threshold) | (prediction == CLASS_TO_ID["Unknown / Ambiguous"])
    automated = ~review
    automated_error = (prediction[automated] != val_true[automated]).mean() if automated.any() else 0.0
    threshold_rows.append({"threshold": threshold, "review_rate": review.mean(), "automated_error_rate": automated_error})
threshold_table = pd.DataFrame(threshold_rows)
eligible = threshold_table.query("automated_error_rate <= 0.10")
selected_threshold = float((eligible if len(eligible) else threshold_table).sort_values("review_rate").iloc[0].threshold)
display(threshold_table)
print(f"Validation-selected threshold: {selected_threshold:.2f}")

reference = clean_results[reference_name]
failure_context = "clean"
confidence = reference["probs"].max(axis=1)
wrong = reference["y_pred"] != reference["y_true"]
if not wrong.any():
    reference = shift_results[reference_name]
    failure_context = "shifted"
    confidence = reference["probs"].max(axis=1)
    wrong = reference["y_pred"] != reference["y_true"]
failure_order = np.argsort(np.where(wrong, confidence, -1))[::-1]
failure_indices = [index for index in failure_order if wrong[index]][: min(8, int(wrong.sum()))]
if failure_indices:
    columns = 4
    rows = int(np.ceil(len(failure_indices) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, index in zip(axes.flat, failure_indices):
        record = splits["test"].iloc[index]
        failure_image = Image.open(record.path).convert("RGB")
        if failure_context == "shifted":
            failure_image = deployment_shift(failure_image)
        axis.imshow(failure_image)
        axis.set_title(
            f"true: {CLASS_NAMES[reference['y_true'][index]]}\npred: {CLASS_NAMES[reference['y_pred'][index]]}\nconf: {confidence[index]:.2f}",
            fontsize=8,
        )
        axis.axis("off")
    fig.suptitle(f"Highest-confidence {failure_context} held-out failures", y=1.02)
    plt.tight_layout()
else:
    print("No clean or shifted errors; add a harder source slice instead of declaring the system solved.")

In [ ]:
reference = clean_results[reference_name]
selected_prediction = reference["y_pred"]
selected_review = (
    (reference["probs"].max(axis=1) < selected_threshold)
    | (selected_prediction == CLASS_TO_ID["Unknown / Ambiguous"])
)
automated = ~selected_review
policy_summary = {
    "threshold": selected_threshold,
    "review_rate": float(selected_review.mean()),
    "automated_count": int(automated.sum()),
    "automated_error_rate": float((selected_prediction[automated] != reference["y_true"][automated]).mean()) if automated.any() else 0.0,
    "defects_sent_to_review": int(((reference["y_true"] != 0) & selected_review).sum()),
}
display(pd.Series(policy_summary, name="held_out_policy").to_frame())

## 12. Enterprise decision and saved evidence

A model comparison does not authorize deployment. The notebook always recommends a limited assistive pilot because the dataset is synthetic and the shift study is narrow. The evidence can still identify a *candidate* for further validation.

In [ ]:
candidate = comparison.iloc[0]
decision = {
    "course": "Modern Computer Vision Foundations",
    "scenario": "five-class enterprise visual quality inspection",
    "data_scope": "deterministic synthetic images; Factory_C held out",
    "candidate_model": candidate["model"],
    "recommendation": "limited assistive pilot with mandatory human review; no autonomous deployment",
    "evidence": {
        "clean_macro_f1": round(float(candidate["clean_macro_f1"]), 4),
        "shift_macro_f1": round(float(candidate["shift_macro_f1"]), 4),
        "clean_defect_recall": round(float(candidate["clean_defect_recall"]), 4),
        "shift_defect_recall": round(float(candidate["shift_defect_recall"]), 4),
        "f1_drop": round(float(candidate["f1_drop"]), 4),
        "reference_policy": policy_summary,
    },
    "risk_boundaries": [
        "Synthetic data does not establish real defect prevalence or capture diversity.",
        "The stress test covers one constructed degradation, not the deployment distribution.",
        "Probabilities are not calibrated and thresholds are not cost-validated.",
        "Unknown/Ambiguous and low-confidence cases require human review.",
    ],
    "required_next_steps": [
        "Acquire licensed representative data with source, time, and product lineage.",
        "Define defect-miss costs and reviewer capacity with domain owners.",
        "Repeat grouped and temporal validation across facilities and devices.",
        "Calibrate scores and validate a selective prediction policy.",
        "Run shadow mode with outcome monitoring, rollback triggers, and audit logs.",
    ],
    "monitor": [
        "input validity and source mix",
        "embedding and confidence drift",
        "class, abstention, and human-override rates",
        "per-source defect recall and delayed outcomes",
        "latency, errors, and model/preprocessing versions",
    ],
    "configuration": asdict(CFG),
    "versions": versions.to_dict(),
}
decision_path = ARTIFACT_DIR / "enterprise_decision.json"
decision_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))
print("Saved:", decision_path)

## 13. Interpret, extend, and transition forward

Answer before closing the notebook:

1. Which model won on clean macro F1? Did it also win under shift?
2. Which class and source produced the most costly errors?
3. Did the nearest neighbours appear to use defect evidence or background style?
4. How much review load did the validation-selected threshold create?
5. What additional evidence would change the recommendation from assistive pilot to automated action?

### Extension experiments

- add a perceptual-hash or embedding near-duplicate detector;
- balance backgrounds across labels and rerun the shortcut experiment;
- calibrate the chosen model with validation-only temperature scaling;
- replace the classifier with a segmentation or anomaly-localisation output contract;
- compare a DINO, CLIP/SigLIP, or other licensed encoder using its official SDK and preprocessing;
- export one candidate through ONNX Runtime and compare numerical and latency drift.

### Where modern vision goes next

Pretrained CNNs are the bridge, not the endpoint. Vision transformers, self-supervised encoders, promptable segmentation, open-vocabulary image-text models, video models, spatial reconstruction, and vision-language-action systems reuse the same foundation developed here: explicit contracts, verified data, transferable representations, stress tests, abstention, and monitored decisions.

Return to the [course README](README.md) for the state-of-the-art map, tooling review, production upgrade path, and references.